In [1]:
import time
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
mn = fetch_openml("mnist_784", version=1, as_frame=False)
X = mn["data"].astype(np.float32) / 255.0
y = mn["target"].astype(int)

X = StandardScaler().fit_transform(X)

X_t = torch.tensor(X, dtype=torch.float32).to(device)
y_t = torch.tensor(y, dtype=torch.long).to(device)

In [3]:
class NetMC(nn.Module):
    def __init__(self, D, H, C):
        super().__init__()
        self.fc1 = nn.Linear(D, H)
        self.fc2 = nn.Linear(H, C)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

model = NetMC(X_t.shape[1], 128, 10).to(device)

In [4]:
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

In [5]:
log_dir = f"logs/mnist/{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

In [6]:
writer.add_graph(model, X_t[:1])

In [7]:
epochs = 5
t0 = time.perf_counter()

In [8]:
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    logits = model(X_t)
    loss = criterion(logits, y_t)

    loss.backward()
    optimizer.step()

    preds = logits.argmax(dim=1)
    acc = (preds == y_t).float().mean().item()

    # TensorBoard scalars
    writer.add_scalar("Loss/train", loss.item(), epoch)
    writer.add_scalar("Accuracy/train", acc, epoch)

    # TensorBoard histograms
    for name, param in model.named_parameters():
        writer.add_histogram(name, param, epoch)

    print(
        f"[MNIST][Epoch {epoch}/{epochs}] "
        f"loss={loss.item():.4f} acc={acc:.4f} "
        f"elapsed={time.perf_counter() - t0:0.1f}s"
    )

[MNIST][Epoch 1/5] loss=2.3358 acc=0.0753 elapsed=0.2s
[MNIST][Epoch 2/5] loss=2.2353 acc=0.2020 elapsed=0.2s
[MNIST][Epoch 3/5] loss=2.1409 acc=0.3759 elapsed=0.2s
[MNIST][Epoch 4/5] loss=2.0507 acc=0.4909 elapsed=0.3s
[MNIST][Epoch 5/5] loss=1.9638 acc=0.5716 elapsed=0.3s


In [9]:
writer.close()

In [10]:
%load_ext tensorboard
%tensorboard --logdir logs/mnist